In [3]:
from pathlib import Path
import polars as pl
pl.Config.set_tbl_cols(-1)          # 显示全部列
pl.Config.set_tbl_width_chars(-1)   # 不限制表格总宽度

polars.config.Config

In [4]:
DATE = "20260624"
ROOT = Path("/data/xujiayi/xjy/l2")
EXCHANGE = "sz"
LEVELS = 10

# 浮点价格允许的误差
PRICE_TOLERANCE = 1e-8

# 数量理论上应完全一致；如有需要可改成1等其他值
QTY_TOLERANCE = 0.0

In [5]:
def null_safe_close(
    left: pl.Expr,
    right: pl.Expr,
    tolerance: float,
) -> pl.Expr:
    """两个值都为空，或者数值误差不超过 tolerance，即认为一致。"""
    return (
        (left.is_null() & right.is_null())
        | ((left - right).abs() <= tolerance)
    ).fill_null(False)


def to_float(column: str) -> pl.Expr:
    """兼容数字列以及包含双引号的字符串数字列。"""
    return (
        pl.col(column)
        .cast(pl.String)
        .str.strip_chars('"')
        .cast(pl.Float64, strict=False)
        .alias(column)
    )


In [6]:
# =============================================================================
# 1. 读取自己生成的快照
# =============================================================================

my_path = ROOT / "proc" / DATE / f"{EXCHANGE}shot_1m.pq"

my_shot = (
    pl.read_parquet(my_path)
    .with_columns(
        pl.col("SecurityID").cast(pl.Int64),
        pl.col("BarTime").cast(pl.Time),
    )
    .sort(["SecurityID", "BarTime"])
)

In [ ]:
# =============================================================================
# 2. 读取官方快照
# =============================================================================

official_path = ROOT / "raw" / DATE / f"{EXCHANGE}shot.pq"

official_columns = ["UpdateTime", "SecurityID"]

for level in range(1, LEVELS + 1):
    official_columns.extend([
        f"BidPrice{level}",
        f"BidVolume{level}",
        f"AskPrice{level}",
        f"AskVolume{level}",
    ])

official_shot = (
    pl.read_parquet(official_path)
    .filter(pl.col("MDStreamID") == 10)
    .select(official_columns)
    .with_columns(
        pl.col("SecurityID").cast(pl.Int64),
        pl.col("UpdateTime").str.to_time(),
    )
    .with_columns([
        to_float(column)
        for column in official_columns
        if column not in {"UpdateTime", "SecurityID"}
    ])
    .sort(["SecurityID", "UpdateTime"])
)

In [ ]:
# =============================================================================
# 3. 将官方快照对齐到自己生成的 BarTime
#
# 对每只证券、每个bar，选择：
#     UpdateTime <= BarTime
# 的最后一条官方快照。
# =============================================================================

bar_grid = (
    my_shot
    .select(["SecurityID", "BarTime"])
    .unique()
    .sort(["SecurityID", "BarTime"])
)

official_aligned = (
    bar_grid
    .join_asof(
        official_shot,
        left_on="BarTime",
        right_on="UpdateTime",
        by="SecurityID",
        strategy="backward",
        check_sortedness=False,
    )
    .with_columns(
        (
            (
                pl.col("BarTime").cast(pl.Int64)
                - pl.col("UpdateTime").cast(pl.Int64)
            )
            / 1_000_000_000
        ).alias("OfficialTimeGapSeconds")
    )
)

In [ ]:
official_aligned.filter(pl.col('SecurityID') == 1).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(15,0))
)

SecurityID,BarTime,UpdateTime,BidPrice1,BidVolume1,AskPrice1,AskVolume1,BidPrice2,BidVolume2,AskPrice2,AskVolume2,BidPrice3,BidVolume3,AskPrice3,AskVolume3,BidPrice4,BidVolume4,AskPrice4,AskVolume4,BidPrice5,BidVolume5,AskPrice5,AskVolume5,BidPrice6,BidVolume6,AskPrice6,AskVolume6,BidPrice7,BidVolume7,AskPrice7,AskVolume7,BidPrice8,BidVolume8,AskPrice8,AskVolume8,BidPrice9,BidVolume9,AskPrice9,AskVolume9,BidPrice10,BidVolume10,AskPrice10,AskVolume10,OfficialTimeGapSeconds
i64,time,time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,09:30:00,09:30:00,10.67,1900.0,10.68,172700.0,10.66,149800.0,10.69,900.0,10.65,292100.0,10.7,2000.0,10.64,136200.0,10.71,98100.0,10.63,127300.0,10.72,45100.0,10.62,80900.0,10.73,120900.0,10.61,256900.0,10.74,5100.0,10.6,209000.0,10.75,81600.0,10.59,82300.0,10.76,41000.0,10.58,30000.0,10.77,74200.0,0.0
1,09:31:00,09:31:00,10.65,392900.0,10.66,32600.0,10.64,211800.0,10.67,79200.0,10.63,195300.0,10.68,82500.0,10.62,164000.0,10.69,57400.0,10.61,408400.0,10.7,7500.0,10.6,439400.0,10.71,63600.0,10.59,97600.0,10.72,50000.0,10.58,90300.0,10.73,130900.0,10.57,127300.0,10.74,7900.0,10.56,99400.0,10.75,88000.0,0.0
1,09:32:00,09:32:00,10.67,12000.0,10.68,1200.0,10.66,28600.0,10.69,23400.0,10.65,105600.0,10.7,65100.0,10.64,152500.0,10.71,40500.0,10.63,213700.0,10.72,52200.0,10.62,255500.0,10.73,136800.0,10.61,372800.0,10.74,14100.0,10.6,462300.0,10.75,91600.0,10.59,99500.0,10.76,61100.0,10.58,101300.0,10.77,83200.0,0.0
1,09:33:00,09:33:00,10.68,15200.0,10.69,53700.0,10.67,6300.0,10.7,67200.0,10.66,181900.0,10.71,25500.0,10.65,173700.0,10.72,52500.0,10.64,150700.0,10.73,137900.0,10.63,245300.0,10.74,18100.0,10.62,252400.0,10.75,101300.0,10.61,373100.0,10.76,63100.0,10.6,500200.0,10.77,85200.0,10.59,100500.0,10.78,230100.0,0.0
1,09:34:00,09:34:00,10.69,2300.0,10.7,85000.0,10.68,92500.0,10.71,52300.0,10.67,183700.0,10.72,54300.0,10.66,223600.0,10.73,139900.0,10.65,212400.0,10.74,28600.0,10.64,167200.0,10.75,111400.0,10.63,252700.0,10.76,66100.0,10.62,256000.0,10.77,85200.0,10.61,436400.0,10.78,232700.0,10.6,517200.0,10.79,207600.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,14:56:00,14:56:00,10.52,95600.0,10.53,29600.0,10.51,1.0419e6,10.54,160900.0,10.5,1.6335e6,10.55,254200.0,10.49,153900.0,10.56,60100.0,10.48,406600.0,10.57,154300.0,10.47,125500.0,10.58,131700.0,10.46,79300.0,10.59,111900.0,10.45,376100.0,10.6,119500.0,10.44,54100.0,10.61,56900.0,10.43,191800.0,10.62,121800.0,0.0
1,14:57:00,14:57:00,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0
1,14:58:00,14:57:54,10.54,377823.0,10.54,377823.0,0.0,277.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,6.0


In [25]:
my_shot.filter(pl.col('SecurityID') == 1).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(14,57))
)#.filter(pl.col('BidPrice1')>pl.col('AskPrice1'))

BarTime,SecurityID,BidPrice1,BidQty1,AskPrice1,AskQty1,BidPrice2,BidQty2,AskPrice2,AskQty2,BidPrice3,BidQty3,AskPrice3,AskQty3,BidPrice4,BidQty4,AskPrice4,AskQty4,BidPrice5,BidQty5,AskPrice5,AskQty5,BidPrice6,BidQty6,AskPrice6,AskQty6,BidPrice7,BidQty7,AskPrice7,AskQty7,BidPrice8,BidQty8,AskPrice8,AskQty8,BidPrice9,BidQty9,AskPrice9,AskQty9,BidPrice10,BidQty10,AskPrice10,AskQty10
time,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


#### 误差统计

In [9]:
# =============================================================================
# 4. 修改官方字段名，避免和自己的字段重名
# =============================================================================

official_rename = {}

for level in range(1, LEVELS + 1):
    official_rename.update({
        f"BidPrice{level}": f"OfficialBidPrice{level}",
        f"BidVolume{level}": f"OfficialBidQty{level}",
        f"AskPrice{level}": f"OfficialAskPrice{level}",
        f"AskVolume{level}": f"OfficialAskQty{level}",
    })

official_aligned = official_aligned.rename(official_rename)


# =============================================================================
# 5. 合并自己生成的快照和官方快照
# =============================================================================

comparison = my_shot.join(
    official_aligned,
    on=["SecurityID", "BarTime"],
    how="left",
)


In [10]:
# =============================================================================
# 6. 逐档计算差值和是否一致
# =============================================================================

difference_expressions = []
match_expressions = []
match_columns = []

for level in range(1, LEVELS + 1):
    pairs = [
        (
            f"BidPrice{level}",
            f"OfficialBidPrice{level}",
            PRICE_TOLERANCE,
        ),
        (
            f"BidQty{level}",
            f"OfficialBidQty{level}",
            QTY_TOLERANCE,
        ),
        (
            f"AskPrice{level}",
            f"OfficialAskPrice{level}",
            PRICE_TOLERANCE,
        ),
        (
            f"AskQty{level}",
            f"OfficialAskQty{level}",
            QTY_TOLERANCE,
        ),
    ]

    for my_column, official_column, tolerance in pairs:
        diff_column = f"{my_column}Diff"
        match_column = f"{my_column}Match"

        difference_expressions.append(
            (
                pl.col(my_column)
                - pl.col(official_column)
            ).alias(diff_column)
        )

        match_expressions.append(
            null_safe_close(
                pl.col(my_column),
                pl.col(official_column),
                tolerance,
            ).alias(match_column)
        )

        match_columns.append(match_column)

comparison = (
    comparison
    .with_columns(difference_expressions)
    .with_columns(match_expressions)
    .with_columns(
        pl.all_horizontal(match_columns).alias("AllLevelsMatch")
    )
)

In [11]:
# =============================================================================
# 7. 总体匹配率
# =============================================================================

summary_expressions = [
    pl.len().alias("RowCount"),
    pl.col("UpdateTime").is_not_null().sum().alias("OfficialMatchedRows"),
    pl.col("AllLevelsMatch").sum().alias("AllLevelsMatchedRows"),
    pl.col("AllLevelsMatch").mean().alias("AllLevelsMatchRate"),
]

for level in range(1, LEVELS + 1):
    for column in (
        f"BidPrice{level}",
        f"BidQty{level}",
        f"AskPrice{level}",
        f"AskQty{level}",
    ):
        summary_expressions.append(
            pl.col(f"{column}Match")
            .mean()
            .alias(f"{column}MatchRate")
        )

summary = comparison.select(summary_expressions)

print("总体匹配情况：")
print(summary)

总体匹配情况：
shape: (1, 44)
┌──────────┬─────────────────────┬──────────────────────┬────────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬────────────────────┬──────────────────┬─────────────────────┬───────────────────┬─────────────────────┬───────────────────┐
│ RowCount ┆ OfficialMatchedRows ┆ AllLevelsMatchedRows ┆ AllLevelsMatchRate ┆ BidPrice1MatchR

In [12]:
# =============================================================================
# 8. 每一档的匹配率
# =============================================================================

level_summary_rows = []

for level in range(1, LEVELS + 1):
    result = comparison.select(
        pl.lit(level).alias("Level"),
        pl.col(f"BidPrice{level}Match").mean().alias("BidPriceMatchRate"),
        pl.col(f"BidQty{level}Match").mean().alias("BidQtyMatchRate"),
        pl.col(f"AskPrice{level}Match").mean().alias("AskPriceMatchRate"),
        pl.col(f"AskQty{level}Match").mean().alias("AskQtyMatchRate"),
    )

    level_summary_rows.append(result)

level_summary = pl.concat(level_summary_rows)

print("逐档匹配率：")
print(level_summary)

逐档匹配率：
shape: (10, 5)
┌───────┬───────────────────┬─────────────────┬───────────────────┬─────────────────┐
│ Level ┆ BidPriceMatchRate ┆ BidQtyMatchRate ┆ AskPriceMatchRate ┆ AskQtyMatchRate │
│ ---   ┆ ---               ┆ ---             ┆ ---               ┆ ---             │
│ i32   ┆ f64               ┆ f64             ┆ f64               ┆ f64             │
╞═══════╪═══════════════════╪═════════════════╪═══════════════════╪═════════════════╡
│ 1     ┆ 0.859265          ┆ 0.695429        ┆ 0.857697          ┆ 0.690102        │
│ 2     ┆ 0.852656          ┆ 0.799538        ┆ 0.851818          ┆ 0.797243        │
│ 3     ┆ 0.850901          ┆ 0.818792        ┆ 0.85013           ┆ 0.819115        │
│ 4     ┆ 0.849622          ┆ 0.831818        ┆ 0.848646          ┆ 0.832552        │
│ 5     ┆ 0.848837          ┆ 0.839337        ┆ 0.847805          ┆ 0.839727        │
│ 6     ┆ 0.84852           ┆ 0.843475        ┆ 0.84725           ┆ 0.843654        │
│ 7     ┆ 0.84846           ┆ 0.

In [13]:
# =============================================================================
# 9. 找出前两档不一致的记录
# =============================================================================

first_two_match_columns = []

for level in range(1, 3):
    first_two_match_columns.extend([
        f"BidPrice{level}Match",
        f"BidQty{level}Match",
        f"AskPrice{level}Match",
        f"AskQty{level}Match",
    ])

first_two_mismatches = (
    comparison
    .filter(
        ~pl.all_horizontal(first_two_match_columns)
    )
    .select([
        "BarTime",
        "UpdateTime",
        "OfficialTimeGapSeconds",
        "SecurityID",

        "BidPrice1",
        "OfficialBidPrice1",
        "BidPrice1Diff",
        "BidQty1",
        "OfficialBidQty1",
        "BidQty1Diff",

        "AskPrice1",
        "OfficialAskPrice1",
        "AskPrice1Diff",
        "AskQty1",
        "OfficialAskQty1",
        "AskQty1Diff",

        "BidPrice2",
        "OfficialBidPrice2",
        "BidPrice2Diff",
        "BidQty2",
        "OfficialBidQty2",
        "BidQty2Diff",

        "AskPrice2",
        "OfficialAskPrice2",
        "AskPrice2Diff",
        "AskQty2",
        "OfficialAskQty2",
        "AskQty2Diff",
    ])
    .sort(["SecurityID", "BarTime"])
)

print("前两档不一致记录：")
print(first_two_mismatches)

前两档不一致记录：
shape: (484_354, 28)
┌──────────┬────────────┬────────────────────────┬────────────┬───────────┬───────────────────┬───────────────┬─────────┬─────────────────┬─────────────┬───────────┬───────────────────┬───────────────┬─────────┬─────────────────┬─────────────┬───────────┬───────────────────┬───────────────┬─────────┬─────────────────┬─────────────┬───────────┬───────────────────┬───────────────┬─────────┬─────────────────┬─────────────┐
│ BarTime  ┆ UpdateTime ┆ OfficialTimeGapSeconds ┆ SecurityID ┆ BidPrice1 ┆ OfficialBidPrice1 ┆ BidPrice1Diff ┆ BidQty1 ┆ OfficialBidQty1 ┆ BidQty1Diff ┆ AskPrice1 ┆ OfficialAskPrice1 ┆ AskPrice1Diff ┆ AskQty1 ┆ OfficialAskQty1 ┆ AskQty1Diff ┆ BidPrice2 ┆ OfficialBidPrice2 ┆ BidPrice2Diff ┆ BidQty2 ┆ OfficialBidQty2 ┆ BidQty2Diff ┆ AskPrice2 ┆ OfficialAskPrice2 ┆ AskPrice2Diff ┆ AskQty2 ┆ OfficialAskQty2 ┆ AskQty2Diff │
│ ---      ┆ ---        ┆ ---                    ┆ ---        ┆ ---       ┆ ---               ┆ ---           ┆ ---     ┆

In [14]:
# =============================================================================
# 10. 单独检查某只证券、某个bar
# =============================================================================

sample = comparison.filter(
    (pl.col("SecurityID") == 1)
    & (pl.col("BarTime") == pl.time(14, 56))
).select([
    "SecurityID",
    "BarTime",
    "UpdateTime",
    "OfficialTimeGapSeconds",

    "BidPrice1",
    "OfficialBidPrice1",
    "BidQty1",
    "OfficialBidQty1",

    "AskPrice1",
    "OfficialAskPrice1",
    "AskQty1",
    "OfficialAskQty1",

    "BidPrice2",
    "OfficialBidPrice2",
    "BidQty2",
    "OfficialBidQty2",

    "AskPrice2",
    "OfficialAskPrice2",
    "AskQty2",
    "OfficialAskQty2",
])

print("单条样本：")
print(sample)

单条样本：
shape: (1, 20)
┌────────────┬──────────┬────────────┬────────────────────────┬───────────┬───────────────────┬─────────┬─────────────────┬───────────┬───────────────────┬─────────┬─────────────────┬───────────┬───────────────────┬──────────┬─────────────────┬───────────┬───────────────────┬──────────┬─────────────────┐
│ SecurityID ┆ BarTime  ┆ UpdateTime ┆ OfficialTimeGapSeconds ┆ BidPrice1 ┆ OfficialBidPrice1 ┆ BidQty1 ┆ OfficialBidQty1 ┆ AskPrice1 ┆ OfficialAskPrice1 ┆ AskQty1 ┆ OfficialAskQty1 ┆ BidPrice2 ┆ OfficialBidPrice2 ┆ BidQty2  ┆ OfficialBidQty2 ┆ AskPrice2 ┆ OfficialAskPrice2 ┆ AskQty2  ┆ OfficialAskQty2 │
│ ---        ┆ ---      ┆ ---        ┆ ---                    ┆ ---       ┆ ---               ┆ ---     ┆ ---             ┆ ---       ┆ ---               ┆ ---     ┆ ---             ┆ ---       ┆ ---               ┆ ---      ┆ ---             ┆ ---       ┆ ---               ┆ ---      ┆ ---             │
│ i64        ┆ time     ┆ time       ┆ f64                   

#### 上交所原还原逻辑异常

In [8]:
sh = pl.read_parquet('/data/xujiayi/xjy/l2/raw/20260624/sh.pq')
sh

BizIndex,Channel,SecurityID,TickTime,Type,BuyOrderNO,SellOrderNO,Price,Qty,TradeMoney,TickBSFlag,LocalTime,SeqNo
i64,i64,i64,str,str,i64,i64,f64,i64,f64,str,str,i64
1,1,751028,"""06:00:00.270""","""S""",0,0,0.0,0,0.0,"""SUSP""","""09:14:00.213""",1
2,1,751900,"""06:00:00.270""","""S""",0,0,0.0,0,0.0,"""SUSP""","""09:14:00.213""",2
3,1,751004,"""06:00:00.270""","""S""",0,0,0.0,0,0.0,"""SUSP""","""09:14:00.213""",3
4,1,751019,"""06:00:00.270""","""S""",0,0,0.0,0,0.0,"""SUSP""","""09:14:00.213""",4
5,1,751992,"""06:00:00.270""","""S""",0,0,0.0,0,0.0,"""SUSP""","""09:14:00.213""",5
…,…,…,…,…,…,…,…,…,…,…,…,…
46340824,5,688757,"""15:05:01.010""","""S""",0,0,0.0,0,0.0,"""ENDTR""","""15:05:01.167""",251904623
46340825,5,603257,"""15:05:01.010""","""S""",0,0,0.0,0,0.0,"""ENDTR""","""15:05:01.167""",251904624
46340826,5,563900,"""15:05:01.010""","""S""",0,0,0.0,0,0.0,"""ENDTR""","""15:05:01.167""",251904625


In [11]:
sh.filter(pl.col('Type')=='A').filter(pl.col('BizIndex')==28157847).filter(pl.col('SecurityID')==688797)

BizIndex,Channel,SecurityID,TickTime,Type,BuyOrderNO,SellOrderNO,Price,Qty,TradeMoney,TickBSFlag,LocalTime,SeqNo
i64,i64,i64,str,str,i64,i64,f64,i64,f64,str,str,i64
28157847,4,688797,"""14:39:36.320""","""A""",16473013,0,600.0,234,0.0,"""B""","""14:40:04.668""",235080714


In [13]:
sh.filter(pl.col('Type')=='T').filter(pl.col('BizIndex').is_in([28157968,28157969])).filter(pl.col('SecurityID')==688797)

BizIndex,Channel,SecurityID,TickTime,Type,BuyOrderNO,SellOrderNO,Price,Qty,TradeMoney,TickBSFlag,LocalTime,SeqNo
i64,i64,i64,str,str,i64,i64,f64,i64,f64,str,str,i64
28157968,4,688797,"""14:40:04.520""","""T""",16473013,16200169,582.39,200,116478.0,"""N""","""14:40:04.668""",235080835
28157969,4,688797,"""14:40:04.520""","""T""",16473013,15955327,582.39,34,19801.26,"""N""","""14:40:04.668""",235080836


In [16]:
sh.filter(pl.col('Type')=='S').filter(pl.col('SecurityID')==688797).filter(pl.col('TickTime')>="14:30:03.000").filter(pl.col('TickTime')<="14:40:05.000")

BizIndex,Channel,SecurityID,TickTime,Type,BuyOrderNO,SellOrderNO,Price,Qty,TradeMoney,TickBSFlag,LocalTime,SeqNo
i64,i64,i64,str,str,i64,i64,f64,i64,f64,str,str,i64
27221729,4,688797,"""14:30:03.880""","""S""",0,0,0.0,0,0.0,"""SUSP""","""14:30:04.055""",227316343
28158384,4,688797,"""14:40:04.520""","""S""",0,0,0.0,0,0.0,"""TRADE""","""14:40:04.669""",235081251


In [18]:
new_shwt= pl.read_parquet('/data/xujiayi/xjy/l2/proc/20260624/shwt.pq')

print(
    new_shwt.filter(
        pl.col("ApplSeqNum") == 28157847
    ).filter(pl.col('SecurityID')==688797)
)

shape: (1, 8)
┌────────────┬───────────┬────────────┬──────────────┬───────┬──────────┬──────┬─────────────┐
│ ApplSeqNum ┆ ChannelNo ┆ SecurityID ┆ TransactTime ┆ Price ┆ OrderQty ┆ Side ┆ OrderStatus │
│ ---        ┆ ---       ┆ ---        ┆ ---          ┆ ---   ┆ ---      ┆ ---  ┆ ---         │
│ i64        ┆ i64       ┆ i64        ┆ time         ┆ f64   ┆ i64      ┆ i32  ┆ str         │
╞════════════╪═══════════╪════════════╪══════════════╪═══════╪══════════╪══════╪═════════════╡
│ 28157847   ┆ 4         ┆ 688797     ┆ 14:39:36.320 ┆ 600.0 ┆ 234      ┆ 1    ┆ 挂单后成交  │
└────────────┴───────────┴────────────┴──────────────┴───────┴──────────┴──────┴─────────────┘


#### 上交所快照对比

In [7]:
my_shot = pl.read_parquet('/data/xujiayi/xjy/l2/proc/20260624/shshot_1m.pq')
ticks = my_shot['SecurityID'].unique().to_list()

In [8]:
shshot = pl.read_parquet('/data/xujiayi/xjy/l2/raw/20260624/shshot.pq')

official_columns = ["UpdateTime", "SecurityID"]
for level in range(1, LEVELS + 1):
    official_columns.extend([
        f"BidPrice{level}",
        f"BidVolume{level}",
        f"AskPrice{level}",
        f"AskVolume{level}",
    ])

official_shot = (
    shshot
    .filter(pl.col('SecurityID').is_in(ticks))
    .select(official_columns)
    .with_columns(
        pl.col("SecurityID").cast(pl.Int64),
        pl.col("UpdateTime").str.to_time(),
    )
    .with_columns([
        to_float(column)
        for column in official_columns
        if column not in {"UpdateTime", "SecurityID"}
    ])
    .sort(["SecurityID", "UpdateTime"])
)
official_shot

UpdateTime,SecurityID,BidPrice1,BidVolume1,AskPrice1,AskVolume1,BidPrice2,BidVolume2,AskPrice2,AskVolume2,BidPrice3,BidVolume3,AskPrice3,AskVolume3,BidPrice4,BidVolume4,AskPrice4,AskVolume4,BidPrice5,BidVolume5,AskPrice5,AskVolume5,BidPrice6,BidVolume6,AskPrice6,AskVolume6,BidPrice7,BidVolume7,AskPrice7,AskVolume7,BidPrice8,BidVolume8,AskPrice8,AskVolume8,BidPrice9,BidVolume9,AskPrice9,AskVolume9,BidPrice10,BidVolume10,AskPrice10,AskVolume10
time,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
08:45:10,501001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
08:45:40,501001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
08:46:10,501001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
08:46:40,501001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
08:47:10,501001,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
15:37:43,900948,2.72,13863.0,2.729,2000.0,2.719,5248.0,2.73,9200.0,2.718,40500.0,2.731,9400.0,2.717,17500.0,2.732,14000.0,2.716,22400.0,2.733,2100.0,2.715,16100.0,2.734,100.0,2.714,1100.0,2.735,100.0,2.713,500.0,2.736,3800.0,2.712,13200.0,2.738,10100.0,2.711,12100.0,2.739,5400.0
15:38:13,900948,2.72,13863.0,2.729,2000.0,2.719,5248.0,2.73,9200.0,2.718,40500.0,2.731,9400.0,2.717,17500.0,2.732,14000.0,2.716,22400.0,2.733,2100.0,2.715,16100.0,2.734,100.0,2.714,1100.0,2.735,100.0,2.713,500.0,2.736,3800.0,2.712,13200.0,2.738,10100.0,2.711,12100.0,2.739,5400.0
15:38:43,900948,2.72,13863.0,2.729,2000.0,2.719,5248.0,2.73,9200.0,2.718,40500.0,2.731,9400.0,2.717,17500.0,2.732,14000.0,2.716,22400.0,2.733,2100.0,2.715,16100.0,2.734,100.0,2.714,1100.0,2.735,100.0,2.713,500.0,2.736,3800.0,2.712,13200.0,2.738,10100.0,2.711,12100.0,2.739,5400.0


In [9]:
bar_grid = (
    my_shot
    .select(["SecurityID", "BarTime"])
    .unique()
    .sort(["SecurityID", "BarTime"])
)

official_aligned = (
    bar_grid
    .join_asof(
        official_shot,
        left_on="BarTime",
        right_on="UpdateTime",
        by="SecurityID",
        strategy="backward",
        check_sortedness=False,
    )
    .with_columns(
        (
            (
                pl.col("BarTime").cast(pl.Int64)
                - pl.col("UpdateTime").cast(pl.Int64)
            )
            / 1_000_000_000
        ).alias("OfficialTimeGapSeconds")
    )
)

In [14]:
official_aligned.filter(pl.col('SecurityID') ==501001).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(15,0))
).drop('UpdateTime')

SecurityID,BarTime,BidPrice1,BidVolume1,AskPrice1,AskVolume1,BidPrice2,BidVolume2,AskPrice2,AskVolume2,BidPrice3,BidVolume3,AskPrice3,AskVolume3,BidPrice4,BidVolume4,AskPrice4,AskVolume4,BidPrice5,BidVolume5,AskPrice5,AskVolume5,BidPrice6,BidVolume6,AskPrice6,AskVolume6,BidPrice7,BidVolume7,AskPrice7,AskVolume7,BidPrice8,BidVolume8,AskPrice8,AskVolume8,BidPrice9,BidVolume9,AskPrice9,AskVolume9,BidPrice10,BidVolume10,AskPrice10,AskVolume10,OfficialTimeGapSeconds
i64,time,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
501001,09:30:00,1.503,4821.0,1.516,3300.0,1.502,3000.0,1.52,629.0,1.501,13000.0,1.525,6746.0,1.5,4700.0,1.537,6208.0,1.489,300.0,1.54,620.0,1.488,100.0,1.549,69150.0,1.487,12200.0,1.55,1000.0,1.485,14500.0,1.56,14385.0,1.484,5000.0,1.564,400.0,1.483,5000.0,1.565,300.0,20.0
501001,09:31:00,1.503,766.0,1.514,500.0,1.501,1813.0,1.515,12800.0,1.5,4000.0,1.516,2900.0,1.489,300.0,1.52,7529.0,1.488,6000.0,1.521,700.0,1.487,15300.0,1.523,3200.0,1.485,5800.0,1.525,6746.0,1.484,5000.0,1.531,4900.0,1.483,5000.0,1.537,6208.0,1.482,5000.0,1.54,620.0,2.0
501001,09:32:00,1.493,21639.0,1.51,3600.0,1.488,3109.0,1.511,12800.0,1.487,15300.0,1.513,7600.0,1.485,5800.0,1.514,500.0,1.484,5000.0,1.516,4900.0,1.483,5000.0,1.52,2070.0,1.482,5000.0,1.521,700.0,1.481,17800.0,1.523,3200.0,1.48,500.0,1.525,6746.0,1.477,16000.0,1.531,4900.0,17.0
501001,09:33:00,1.503,2180.0,1.508,12800.0,1.497,21500.0,1.509,7600.0,1.496,15000.0,1.51,3700.0,1.495,19500.0,1.514,500.0,1.494,8759.0,1.516,4900.0,1.493,3000.0,1.52,1870.0,1.488,4209.0,1.521,700.0,1.487,15300.0,1.523,3200.0,1.486,1000.0,1.525,6746.0,1.485,5800.0,1.531,4900.0,2.0
501001,09:34:00,1.497,19998.0,1.506,1267.0,1.494,8759.0,1.514,13300.0,1.493,3000.0,1.516,3633.0,1.488,4909.0,1.52,1870.0,1.487,15300.0,1.521,700.0,1.486,1000.0,1.523,3200.0,1.485,5800.0,1.525,6746.0,1.484,5000.0,1.531,4900.0,1.483,5000.0,1.537,6208.0,1.482,5000.0,1.549,28976.0,5.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
501001,14:56:00,1.508,4624.0,1.509,12633.0,1.507,24926.0,1.512,15920.0,1.506,14888.0,1.513,200.0,1.505,1000.0,1.514,5335.0,1.504,3900.0,1.515,1100.0,1.502,2000.0,1.517,4100.0,1.501,10900.0,1.518,4815.0,1.5,3300.0,1.521,700.0,1.498,100.0,1.524,300.0,1.495,2000.0,1.525,3167.0,2.0
501001,14:57:00,1.508,16124.0,1.509,11633.0,1.507,9426.0,1.512,15920.0,1.506,14888.0,1.513,200.0,1.505,1000.0,1.514,5335.0,1.504,3900.0,1.517,4100.0,1.502,1000.0,1.518,4815.0,1.501,10900.0,1.521,700.0,1.5,3300.0,1.524,300.0,1.498,100.0,1.525,3167.0,1.495,2000.0,1.526,3783.0,2.0
501001,14:58:00,1.508,16724.0,1.509,11633.0,1.507,8426.0,1.512,15920.0,1.505,1000.0,1.513,200.0,1.504,3900.0,1.514,5335.0,1.502,1000.0,1.517,4100.0,1.501,10900.0,1.518,4815.0,1.5,3300.0,1.521,700.0,1.498,100.0,1.524,300.0,1.495,2000.0,1.525,3167.0,1.493,100.0,1.526,3783.0,5.0


In [15]:
my_shot.filter(pl.col('SecurityID') ==501001).filter(
    (pl.col('BarTime')>=pl.time(9,30)) & (pl.col('BarTime')<=pl.time(15,0))
)

BarTime,SecurityID,BidPrice1,BidQty1,AskPrice1,AskQty1,BidPrice2,BidQty2,AskPrice2,AskQty2,BidPrice3,BidQty3,AskPrice3,AskQty3,BidPrice4,BidQty4,AskPrice4,AskQty4,BidPrice5,BidQty5,AskPrice5,AskQty5,BidPrice6,BidQty6,AskPrice6,AskQty6,BidPrice7,BidQty7,AskPrice7,AskQty7,BidPrice8,BidQty8,AskPrice8,AskQty8,BidPrice9,BidQty9,AskPrice9,AskQty9,BidPrice10,BidQty10,AskPrice10,AskQty10
time,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
09:30:00,501001,1.503,4821.0,1.516,3300.0,1.502,3000.0,1.52,629.0,1.501,13000.0,1.525,6746.0,1.5,4700.0,1.537,6208.0,1.489,300.0,1.54,620.0,1.488,100.0,1.549,69150.0,1.487,12200.0,1.55,1000.0,1.485,14500.0,1.56,14385.0,1.484,5000.0,1.564,400.0,1.483,5000.0,1.565,300.0
09:31:00,501001,1.488,5179.0,1.514,500.0,1.487,15300.0,1.515,12800.0,1.485,5800.0,1.516,2900.0,1.484,5000.0,1.52,7529.0,1.483,5000.0,1.521,700.0,1.482,5000.0,1.523,3200.0,1.481,17800.0,1.525,6746.0,1.48,500.0,1.531,4900.0,1.477,16000.0,1.537,6208.0,1.46,1700.0,1.54,620.0
09:32:00,501001,1.493,21639.0,1.51,3600.0,1.488,3109.0,1.511,12800.0,1.487,15300.0,1.513,7600.0,1.485,5800.0,1.514,500.0,1.484,5000.0,1.516,4900.0,1.483,5000.0,1.52,2070.0,1.482,5000.0,1.521,700.0,1.481,17800.0,1.523,3200.0,1.48,500.0,1.525,6746.0,1.477,16000.0,1.531,4900.0
09:33:00,501001,1.503,2180.0,1.508,12800.0,1.497,21500.0,1.509,7600.0,1.496,15000.0,1.51,200.0,1.495,19500.0,1.514,500.0,1.494,8759.0,1.516,4900.0,1.493,3000.0,1.52,1870.0,1.488,4209.0,1.521,700.0,1.487,15300.0,1.523,3200.0,1.486,1000.0,1.525,6746.0,1.485,5800.0,1.531,4900.0
09:34:00,501001,1.497,18731.0,1.506,1267.0,1.494,8759.0,1.514,13300.0,1.493,3000.0,1.516,3633.0,1.488,4909.0,1.52,1870.0,1.487,15300.0,1.521,700.0,1.486,1000.0,1.523,3200.0,1.485,5800.0,1.525,6746.0,1.484,5000.0,1.531,4900.0,1.483,5000.0,1.537,6208.0,1.482,5000.0,1.549,28976.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14:56:00,501001,1.508,4624.0,1.509,12633.0,1.507,24926.0,1.512,15920.0,1.506,14888.0,1.513,200.0,1.505,1000.0,1.514,5335.0,1.504,3900.0,1.515,1100.0,1.502,2000.0,1.517,4100.0,1.501,10900.0,1.518,4815.0,1.5,3300.0,1.521,700.0,1.498,100.0,1.524,300.0,1.495,2000.0,1.525,3167.0
14:57:00,501001,1.508,15724.0,1.509,11633.0,1.507,9426.0,1.512,15920.0,1.506,14888.0,1.513,200.0,1.505,1000.0,1.514,5335.0,1.504,3900.0,1.517,4100.0,1.502,1000.0,1.518,4815.0,1.501,10900.0,1.521,700.0,1.5,3300.0,1.524,300.0,1.498,100.0,1.525,3167.0,1.495,2000.0,1.526,3783.0
14:58:00,501001,1.508,16724.0,1.509,11633.0,1.507,8426.0,1.512,15920.0,1.505,1000.0,1.513,200.0,1.504,3900.0,1.514,5335.0,1.502,1000.0,1.517,4100.0,1.501,10900.0,1.518,4815.0,1.5,3300.0,1.521,700.0,1.498,100.0,1.524,300.0,1.495,2000.0,1.525,3167.0,1.493,100.0,1.526,3783.0
